# Cell Painting Parquet Descriptive Statistics

This notebook provides statistical analysis and visualizations of Cell Painting parquet files generated by cytotable.

## Parameters

These parameters can be overridden by Papermill:

In [ ]:
# Papermill parameters
parquet_file = "../nf-runs/test-profile-outdir/cytotable/2021_04_26_Batch1_BR00117035_A01_1.parquet"
n_head_rows = 10
output_prefix = "sample"

## Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
# Load parquet file
print(f"Loading parquet file: {parquet_file}")
df = pd.read_parquet(parquet_file)
print(f"✓ Successfully loaded parquet file")

## First Rows (Head)

In [ ]:
print("=" * 80)
print(f"FIRST {n_head_rows} ROWS (showing first 20 columns)")
print("=" * 80)
display(df.head(n_head_rows).iloc[:, :20])

## Descriptive Statistics

Basic statistical summary of numeric columns (showing first 50 features):

In [ ]:
# Get numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns
print(f"Number of numeric columns: {len(numeric_cols)}")
print()

# Calculate descriptive stats for ALL numeric columns
descriptive_stats = df[numeric_cols].describe().T
descriptive_stats.reset_index(inplace=True)
descriptive_stats.rename(columns={'index': 'feature'}, inplace=True)

# Save to parquet file
stats_filename = parquet_file.replace('.parquet', '_descriptive_stats.parquet')
descriptive_stats.to_parquet(stats_filename, index=False)
print(f"✓ Saved descriptive statistics to: {stats_filename}")
print()

# Show descriptive stats for first 50 numeric columns
print("=" * 80)
print("DESCRIPTIVE STATISTICS (first 50 numeric columns)")
print("=" * 80)
display(descriptive_stats.head(50).set_index('feature'))

## Missing Values Analysis

In [ ]:
# Check for missing values
missing_counts = df.isnull().sum()
missing_cols = missing_counts[missing_counts > 0].sort_values(ascending=False)

print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)
if len(missing_cols) > 0:
    print(f"Columns with missing values: {len(missing_cols)}")
    print(f"\nTop 20 columns with most missing values:")
    print()
    for col, count in missing_cols.head(20).items():
        pct = (count / len(df)) * 100
        print(f"{col:60s}: {count:5d} ({pct:5.1f}%)")
else:
    print("✓ No missing values found in dataset")
print()

## Feature Categories

Breakdown of measurement types across compartments:

In [ ]:
# Categorize features by measurement type
feature_categories = {}
compartments = ['Cells', 'Cytoplasm', 'Nuclei']
measurement_types = ['AreaShape', 'Intensity', 'Texture', 'Granularity', 
                     'Correlation', 'Neighbors', 'RadialDistribution', 'Skeleton']

for compartment in compartments:
    feature_categories[compartment] = {}
    for meas_type in measurement_types:
        pattern = f"{compartment}_{meas_type}"
        matching_cols = [col for col in df.columns if pattern in col]
        if matching_cols:
            feature_categories[compartment][meas_type] = len(matching_cols)

print("=" * 80)
print("FEATURE CATEGORIES BY COMPARTMENT")
print("=" * 80)
for compartment, measurements in feature_categories.items():
    if measurements:
        print(f"\n{compartment}:")
        for meas_type, count in sorted(measurements.items(), key=lambda x: x[1], reverse=True):
            print(f"  {meas_type:25s}: {count:5d} features")

## Quick Visualizations

### Cell Size Distribution

In [ ]:
# Check if we have the required columns for size distribution
has_size_cols = (
    'Cells_AreaShape_Area' in df.columns or 
    'Nuclei_AreaShape_Area' in df.columns or 
    'Cytoplasm_AreaShape_Area' in df.columns
)

if has_size_cols:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Cell area
    if 'Cells_AreaShape_Area' in df.columns:
        axes[0].hist(df['Cells_AreaShape_Area'], bins=50, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Cell Area (pixels)')
        axes[0].set_ylabel('Frequency')
        axes[0].set_title('Cell Size Distribution')
        axes[0].axvline(df['Cells_AreaShape_Area'].median(), color='red', 
                       linestyle='--', label=f'Median: {df["Cells_AreaShape_Area"].median():.0f}')
        axes[0].legend()

    # Nuclear area
    if 'Nuclei_AreaShape_Area' in df.columns:
        axes[1].hist(df['Nuclei_AreaShape_Area'], bins=50, edgecolor='black', alpha=0.7, color='green')
        axes[1].set_xlabel('Nuclear Area (pixels)')
        axes[1].set_ylabel('Frequency')
        axes[1].set_title('Nuclear Size Distribution')
        axes[1].axvline(df['Nuclei_AreaShape_Area'].median(), color='red', 
                       linestyle='--', label=f'Median: {df["Nuclei_AreaShape_Area"].median():.0f}')
        axes[1].legend()

    # Cytoplasm area
    if 'Cytoplasm_AreaShape_Area' in df.columns:
        axes[2].hist(df['Cytoplasm_AreaShape_Area'], bins=50, edgecolor='black', alpha=0.7, color='orange')
        axes[2].set_xlabel('Cytoplasm Area (pixels)')
        axes[2].set_ylabel('Frequency')
        axes[2].set_title('Cytoplasm Size Distribution')
        axes[2].axvline(df['Cytoplasm_AreaShape_Area'].median(), color='red', 
                       linestyle='--', label=f'Median: {df["Cytoplasm_AreaShape_Area"].median():.0f}')
        axes[2].legend()

    plt.tight_layout()
    
    # Save figure to file
    size_dist_filename = f"{output_prefix}_size_distributions.png"
    plt.savefig(size_dist_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved cell size distributions to: {size_dist_filename}")
    
    plt.show()
else:
    print("Note: No area shape columns found for size distribution plots")

### DNA Intensity Distribution

In [ ]:
dna_intensity_col = 'Nuclei_Intensity_MeanIntensity_DNA'
if dna_intensity_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(df[dna_intensity_col], bins=50, edgecolor='black', alpha=0.7, color='blue')
    axes[0].set_xlabel('Mean DNA Intensity')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('DNA Intensity Distribution')
    axes[0].axvline(df[dna_intensity_col].median(), color='red', 
                   linestyle='--', label=f'Median: {df[dna_intensity_col].median():.3f}')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(df[dna_intensity_col].dropna())
    axes[1].set_ylabel('Mean DNA Intensity')
    axes[1].set_title('DNA Intensity Box Plot')
    axes[1].set_xticklabels(['DNA'])
    
    plt.tight_layout()
    
    # Save figure to file
    dna_dist_filename = f"{output_prefix}_dna_intensity.png"
    plt.savefig(dna_dist_filename, dpi=300, bbox_inches='tight')
    print(f"✓ Saved DNA intensity distribution to: {dna_dist_filename}")
    
    plt.show()
else:
    print(f"Note: Column {dna_intensity_col} not found")

## Summary Statistics Table

In [ ]:
# Create summary table of key metrics
summary_data = {
    'Metric': [],
    'Value': []
}

summary_data['Metric'].append('Total Cells')
summary_data['Value'].append(f"{len(df):,}")

if 'Cells_AreaShape_Area' in df.columns:
    summary_data['Metric'].append('Mean Cell Area')
    summary_data['Value'].append(f"{df['Cells_AreaShape_Area'].mean():.2f}")
    summary_data['Metric'].append('Median Cell Area')
    summary_data['Value'].append(f"{df['Cells_AreaShape_Area'].median():.2f}")

if 'Nuclei_AreaShape_Area' in df.columns:
    summary_data['Metric'].append('Mean Nuclear Area')
    summary_data['Value'].append(f"{df['Nuclei_AreaShape_Area'].mean():.2f}")

if dna_intensity_col in df.columns:
    summary_data['Metric'].append('Mean DNA Intensity')
    summary_data['Value'].append(f"{df[dna_intensity_col].mean():.4f}")

summary_df = pd.DataFrame(summary_data)

print("=" * 80)
print("SUMMARY STATISTICS")
print("=" * 80)
display(summary_df)

## Report Complete

This descriptive statistics report was generated using Papermill for automated analysis of Cell Painting parquet files.